# CDC-4 — The CDC event envelope

**Break → Detect → Fix → Prove.** Every Debezium change is an **envelope**, not a row:
`before`, `after`, `op` (`c` create / `r` read-snapshot / `u` update / `d` delete), `ts_ms`,
and `source` (with the WAL `lsn`). We read one event of each op to see the envelope's shape,
then register a **second** connector with the `ExtractNewRecordState` SMT to **flatten** that
envelope into a plain row on a different topic — and compare the two contracts side by side.

**Prerequisite:** `make cdc-up` (Postgres + Kafka Connect; see the [track README](../README.md)).
**Laptop-safe:** one 20-row table, three DML statements, bounded topic reads, teardown of **both**
connectors + the table + both topics. **Connect-safe:** `cdc_helpers` + `kafka-python` only.

> **Isolation:** table `cdc4_orders`; connectors `cdc4-orders` (full envelope, prefix `dbz`) and
> `cdc4-unwrap` (flattened, prefix `dbz4u`). The next cell tears down any leftovers so re-runs start clean.

In [ ]:
from common import cdc_helpers as cdc
from kafka import KafkaAdminClient
import json, time

TABLE        = "cdc4_orders"
ENV_CONN     = "cdc4-orders"      # full-envelope connector  -> topic dbz.public.cdc4_orders
UNWRAP_CONN  = "cdc4-unwrap"      # ExtractNewRecordState SMT -> topic dbz4u.public.cdc4_orders
ENV_TOPIC    = cdc.topic_name(TABLE)                  # "dbz.public.cdc4_orders"
UNWRAP_TOPIC = f"dbz4u.public.{TABLE}"                # distinct topic.prefix below

print("Connect REST up:", cdc.connect_up(timeout=40))

def drop_unwrap_topic():
    """cdc.teardown only knows the default 'dbz' prefix, so delete the dbz4u topic explicitly."""
    admin = KafkaAdminClient(bootstrap_servers=cdc.BOOTSTRAP)
    try:
        admin.delete_topics([UNWRAP_TOPIC])
    except Exception:
        pass
    finally:
        admin.close()

# Idempotent clean slate: tear down BOTH connectors (+ slots + topics) and the table.
cdc.teardown(ENV_CONN, TABLE)                 # connector + slot + table + dbz topic
cdc.delete_connector(UNWRAP_CONN)             # second connector...
time.sleep(2); cdc.drop_slot(cdc._safe_slot(UNWRAP_CONN))   # ...and its slot
drop_unwrap_topic()                           # ...and its dbz4u topic
print("clean slate: both connectors + table + both topics removed")

## 1. Seed the source table

A normal 20-row OLTP table. `replica_identity_full=False` keeps Postgres' **default** replica
identity — so an UPDATE's `before` will contain only the **primary key**, not the old column
values. Seeing that partial-`before` gotcha (and its fix, `REPLICA IDENTITY FULL`) is **CDC-6**.

In [ ]:
cdc.seed_orders(TABLE, n=20, replica_identity_full=False)
print("seeded public.%s:" % TABLE,
      cdc.pg_exec(f"SELECT count(*), min(id), max(id) FROM public.{TABLE}", fetch=True)[0])

## 2. Register the envelope connector (`snapshot.mode=initial`)

`common.cdc_helpers.debezium_pg_config` fills the standard defaults (pgoutput, JSON schemas off,
`decimal.handling.mode=double`, `tombstones.on.delete=true`). With `snapshot.mode=initial` the
connector emits one `r` (read) event per existing row, then streams live changes.

In [ ]:
env_cfg = cdc.debezium_pg_config(ENV_CONN, TABLE, snapshot_mode="initial")
print("register ->", cdc.register_connector(ENV_CONN, env_cfg)["status"])   # 201
print("state    ->", cdc.wait_for_connector(ENV_CONN, timeout=60))          # RUNNING

## 3. Break/observe — drive one of each op

The 20-row snapshot already produced `r` events. Now run an INSERT, an UPDATE, and a DELETE
directly in Postgres to produce `c`, `u`, and `d` (the delete also emits a **tombstone** —
a null-value record). `time.sleep(5)` covers the small WAL-decode lag, then drain the topic.

In [ ]:
cdc.pg_exec(f"INSERT INTO public.{TABLE}(id,customer,amount,status) VALUES (200,'envelope-demo',42.00,'NEW')")  # c
cdc.pg_exec(f"UPDATE public.{TABLE} SET status='SHIPPED', amount=43.00 WHERE id=200")                                # u
cdc.pg_exec(f"DELETE FROM public.{TABLE} WHERE id=200")                                                              # d (+ tombstone)
time.sleep(5)

events = cdc.read_cdc_events(ENV_TOPIC)
print("total events:", len(events))

## 4. Detect — op counts across the topic

Every op should appear, plus the tombstone: snapshot reads (`r`), the live insert (`c`), the
update (`u`), the delete (`d`), and the null-value `tombstone`.

In [ ]:
print("op counts:", cdc.op_counts(events))   # e.g. {'r': 20, 'c': 1, 'u': 1, 'd': 1, 'tombstone': 1}

## 5. The envelope shape — one event of each op

Pretty-print the `before`/`after`/`op` of one event per op. Note the pairing:

- **`c`** create  → `before=null`, `after={the new row}`
- **`u`** update  → `before` is **only the PK** (default replica identity), `after={the new row}`
- **`d`** delete  → `before={the row image}`, `after=null`
- **tombstone**   → the whole message **value is `null`** (op parsed as `None`) — the marker a
  log-compacted topic uses to physically drop the key.

In [ ]:
def first(op):
    return next((e for e in events if e["op"] == op), None)

for op in ("c", "u", "d"):
    e = first(op)
    if e is None:
        print(f"[{op}] (not found)\n"); continue
    print(f"[{op}] before/after:")
    print(json.dumps({k: e["raw"].get(k) for k in ("before", "after", "op")}, indent=2, default=str))
    print()

tomb = next((e for e in events if e["op"] is None), None)
print("[tombstone] raw value is:", tomb["raw"] if tomb else "(none)", "(null-value record)")

## 6. Ordering metadata — `ts_ms` and `source.lsn`

The fields that make CDC *correct* aren't `before`/`after` — they're the **ordering metadata** on
the raw envelope. `source.lsn` (the WAL log-sequence number) is monotonic in commit order, so it,
not wall-clock `ts_ms`, is the tiebreaker a sink uses to apply changes in order and de-duplicate on
replay. This is the seam into the idempotent **upsert/MERGE** of CDC-7.

In [ ]:
for op in ("c", "u", "d"):
    e = first(op)
    if e is None:
        continue
    raw = e["raw"]
    src = raw.get("source", {}) or {}
    print(f"op={op}  ts_ms={raw.get('ts_ms')}  lsn={src.get('lsn')}  table={src.get('table')}")

## 7. Fix/route — flatten with `ExtractNewRecordState` (the unwrap SMT)

Register a **second** connector over the **same table**, adding the transform and a **distinct
`topic.prefix=dbz4u`** so it publishes to its own topic (`dbz4u.public.cdc4_orders`) — no collision
with the envelope topic. The SMT options:

- `transforms.unwrap.type = io.debezium.transforms.ExtractNewRecordState` — emit the `after` row directly
- `transforms.unwrap.drop.tombstones = false` — keep tombstones (compaction needs them)
- `transforms.unwrap.delete.handling.mode = rewrite` — a delete becomes a row + `__deleted` marker
  (instead of being dropped), so a flat sink can still react to it.

In [ ]:
unwrap_cfg = cdc.debezium_pg_config(
    UNWRAP_CONN, TABLE, snapshot_mode="initial",
    extra={
        "topic.prefix": "dbz4u",                                # -> dbz4u.public.cdc4_orders
        "transforms": "unwrap",
        "transforms.unwrap.type": "io.debezium.transforms.ExtractNewRecordState",
        "transforms.unwrap.drop.tombstones": "false",
        "transforms.unwrap.delete.handling.mode": "rewrite",
    },
)
print("register ->", cdc.register_connector(UNWRAP_CONN, unwrap_cfg)["status"])   # 201
print("state    ->", cdc.wait_for_connector(UNWRAP_CONN, timeout=60))             # RUNNING

# Re-drive the SAME change sequence so the unwrap topic carries c/u/d for the same key.
cdc.pg_exec(f"INSERT INTO public.{TABLE}(id,customer,amount,status) VALUES (201,'unwrap-demo',7.00,'NEW')")
cdc.pg_exec(f"UPDATE public.{TABLE} SET status='SHIPPED' WHERE id=201")
cdc.pg_exec(f"DELETE FROM public.{TABLE} WHERE id=201")
time.sleep(5)

## 8. Prove — raw envelope vs unwrapped, side by side

`read_cdc_events` parses `before`/`after`/`op` out of the **envelope** schema, so on the
**unwrapped** topic those come back `None` — there is no wrapper. So we read the unwrap topic with a
raw `KafkaConsumer` and print the **value as-is**: it's the flattened row (the `after` fields
directly), and a delete is a row carrying `__deleted:"true"`.

In [ ]:
from kafka import KafkaConsumer

def read_raw_values(topic, max_ms=12000):
    c = KafkaConsumer(topic, bootstrap_servers=cdc.BOOTSTRAP, auto_offset_reset="earliest",
                      enable_auto_commit=False, consumer_timeout_ms=max_ms,
                      value_deserializer=lambda b: b.decode() if b else None)
    out = []
    try:
        for m in c:
            out.append(None if m.value is None else json.loads(m.value))
    finally:
        c.close()
    return out

unwrapped = read_raw_values(UNWRAP_TOPIC)
print("unwrapped record count:", len(unwrapped))
print("flattened values (no before/after wrapper):")
for v in unwrapped:
    print("  ", "TOMBSTONE (null value)" if v is None else v)

## 9. Side by side — same change, two contracts

The envelope topic carries the full change event; the unwrapped topic carries the row a simple
sink consumes. Same source DML, two payload shapes — that **is** the Prove-it here.

In [ ]:
env_c = first("c")
env_d = first("d")
print("ENVELOPE  create value:")
print(json.dumps({k: env_c["raw"].get(k) for k in ("before","after","op","ts_ms")}, indent=2, default=str) if env_c else "  (none)")
print("\nUNWRAPPED create value (the after row directly):")
print("  ", next((v for v in unwrapped if v and v.get("id") == 201 and v.get("__deleted") in (None, "false")), "(none)"))

print("\nENVELOPE  delete value:")
print(json.dumps({k: env_d["raw"].get(k) for k in ("before","after","op")}, indent=2, default=str) if env_d else "  (none)")
print("\nUNWRAPPED delete value (row + __deleted marker):")
print("  ", next((v for v in unwrapped if v and v.get("__deleted") == "true"), "(none)"))

print("\nTakeaway: unwrap for simple current-state sinks; keep the full envelope when you need")
print("op/before for an idempotent MERGE upsert (CDC-7). Topic name = <prefix>.<schema>.<table>.")

## 10. Teardown

Tear down **both** connectors, the table, and **both** topics. `cdc.teardown` handles the envelope
connector (+ slot + table + the `dbz` topic); the unwrap connector's slot and its distinct `dbz4u`
topic are removed explicitly (`teardown`'s `drop_topic` only knows the default `dbz` prefix).

In [ ]:
cdc.delete_connector(UNWRAP_CONN)
time.sleep(2); cdc.drop_slot(cdc._safe_slot(UNWRAP_CONN))
drop_unwrap_topic()
cdc.teardown(ENV_CONN, TABLE)     # connector + slot + table + dbz topic
print("slots now:", cdc.list_slots(), "| make clean clears anything left under .tmp/")